# 02 - 数据结构（Java vs Python）

只讲和 Java 有差异、且你写 AI 应用会高频用到的点。


## 今日目标（20-30 分钟）

- 掌握 `list / dict / set / tuple` 的核心差异
- 会用解包（unpacking）快速写出更 Pythonic 的代码
- 会做常见结构转换，避免写冗长样板代码

完成标准：你能不用查资料写出一个“统计词频 + 去重 + 排序输出”的小段脚本。


## 1. list（对比 Java 的 ArrayList）


**list 是 Python 最常用的集合类型**，对应 Java 的 `ArrayList<T>`。

核心特点：
- 有序、可重复、可修改
- 用 `[]` 创建，支持混合类型（但实际开发中一般存同一种类型）
- 独有的**切片语法** `list[start:end:step]`，是 Java 没有的强力特性
- 赋值是引用传递（和 Java 对象一样），要复制用 `.copy()`

In [ ]:
# 说明：本段示例对应「1. list（对比 Java 的 ArrayList）」，建议先看注释再运行，重点观察输出与预期是否一致。

# Java: List<String> names = new ArrayList<>();
names = ["alice", "bob", "cindy"]

print(names[0])
print(names[-1])
print(len(names))


alice
cindy
3


In [ ]:
# 说明：本段示例对应「1. list（对比 Java 的 ArrayList）」，建议先看注释再运行，重点观察输出与预期是否一致。

nums = [10, 20, 30, 40, 50]

# [Review] 切片是 Java 没有的语法，每行加注释说明含义
print(nums[1:4])   # [20, 30, 40] 索引1到3（不含4），类比 subList(1, 4)
print(nums[:3])    # [10, 20, 30] 从头取3个
print(nums[3:])    # [40, 50] 从索引3到末尾
print(nums[::2])   # [10, 30, 50] 步长为2，隔一个取一个
print(nums[::-1])  # [50, 40, 30, 20, 10] 反转整个列表


In [ ]:
# 说明：演示 list 的三种常见写操作：append 追加单个、extend 批量追加、insert 指定位置插入。

items = ["a", "b"]
items.append("c")
items.extend(["d", "e"])
items.insert(1, "x")
print(items)

items.remove("x")
last = items.pop()
print(items, last)


['a', 'x', 'b', 'c', 'd', 'e']
['a', 'b', 'c', 'd'] e


In [ ]:
# 说明：本段示例对应「1. list（对比 Java 的 ArrayList）」，建议先看注释再运行，重点观察输出与预期是否一致。

a = [1, 2, 3]
b = a
b.append(4)
print("a =", a, "| b =", b)

# 这里显式 copy：避免后续修改互相污染，真实业务里是高频坑点
c = a.copy()
c.append(5)
print("a =", a, "| c =", c)


## 2. dict（对比 Java 的 HashMap）


**dict 是 Python 的键值对集合**，对应 Java 的 `HashMap<K, V>`。

核心特点：
- 用 `{}` 加 `key: value` 创建，key 必须是不可变类型（str、int、tuple）
- Python 3.7+ 保证插入顺序（Java 的 `LinkedHashMap` 才有顺序）
- 取值推荐用 `.get(key, 默认值)`，避免 key 不存在时抛 `KeyError`
- AI 开发中大量用 dict 传递结构化数据（如 API 请求体、配置项）

In [1]:
# 说明：本段示例对应「2. dict（对比 Java 的 HashMap）」，建议先看注释再运行，重点观察输出与预期是否一致。

# Java: Map<String, Integer> score = new HashMap<>();
score = {"math": 95, "english": 88}

print(score["math"])
print(score.get("science"))       # None，不报错
print(score.get("science", 0))    # 0，返回默认值
print("math" in score)

# 💡 实战：调用外部 API 返回的 JSON 字段不一定都存在，
# 直接用 score["xxx"] 取不存在的 key 会抛 KeyError 导致接口 500
# 必须用 .get() 防御性取值，这是 AI 应用开发里最高频的写法之一
# response.get("choices", [])[0].get("message", {}).get("content", "")


95
None
0
True


In [2]:
# 说明：本段示例对应「2. dict（对比 Java 的 HashMap）」，建议先看注释再运行，重点观察输出与预期是否一致。

user = {"name": "kai", "age": 18}
user["city"] = "shanghai"
user["age"] = 19

for k, v in user.items():
    print(k, "=>", v)


name => kai
age => 19
city => shanghai


In [3]:
# 说明：演示用 dict.get(key, default) 做计数聚合，避免首次出现 key 时抛异常。

logs = [
    {"level": "INFO", "msg": "start"},
    {"level": "ERROR", "msg": "timeout"},
    {"level": "INFO", "msg": "retry"},
]

counter = {}
for log in logs:
    level = log["level"]
    # 用 get + 0 聚合计数，避免 key 第一次出现时报错
    counter[level] = counter.get(level, 0) + 1

print(counter)

# 💡 实战：这个计数模式很常见，但标准库有更简洁的方案
# from collections import Counter
# counter = Counter(log["level"] for log in logs)
# 一行替代整段循环，生产代码里优先用 Counter


{'INFO': 2, 'ERROR': 1}


## 3. set（对比 Java 的 HashSet）


### 创建方式横向对比（重点看空花括号）

| 类型 | 常见创建方式 | 空字面量写法 |
|---|---|---|
| `list` | `[1, 2, 3]` / `list(iterable)` | `[]` |
| `tuple` | `(1, 2, 3)` / `tuple(iterable)` | `()` |
| `dict` | `{'k': 'v'}` / `dict(k='v')` | `{}` ✅ |
| `set` | `{1, 2, 3}` / `set(iterable)` | `set()` ✅（不是 `{}`） |

结论：`{}` 在 Python 里**永远是空 dict**；空 set 只能写 `set()`。


In [ ]:
# 说明：用 type() 直接验证各类集合创建结果，避免把 {} 误当成空 set。

a = {}
b = set()
c = {1, 2, 3}
d = {'name': 'kai'}

print(type(a), a)
print(type(b), b)
print(type(c), c)
print(type(d), d)


**set 是无序、不重复的集合**，对应 Java 的 `HashSet<T>`。

核心特点：
- 用 `set()` 或 `{1, 2, 3}` 创建（注意：`{}` 空花括号是 dict，不是 set）
- 最常见用途：**去重**，直接 `set(list)` 一行搞定
- 支持数学集合运算：`|` 并集、`&` 交集、`-` 差集
- 无序，不能用索引取值

In [ ]:
# 说明：本段示例对应「3. set（对比 Java 的 HashSet）」，建议先看注释再运行，重点观察输出与预期是否一致。

ids = [101, 102, 101, 103, 102]
unique_ids = set(ids)
print(unique_ids)
print(101 in unique_ids)

# ⚠️ 坑：set 不保证顺序，每次运行输出顺序可能不同
# 如果去重后还需要排序或保持原来的顺序，要额外处理：
# sorted(set(ids))            → 去重 + 排序
# list(dict.fromkeys(ids))    → 去重 + 保持原有顺序（Python 3.7+）


In [ ]:
# 说明：演示集合并集/交集/差集运算，便于处理去重、标签和权限等集合场景。

a = {1, 2, 3, 4}
b = {3, 4, 5}

# [Review] set 运算符对 Java 开发者不直观，加注释说明
print(a | b)  # 并集 {1,2,3,4,5}（Java: SetUtils.union）
print(a & b)  # 交集 {3,4}（Java: retainAll）
print(a - b)  # 差集 {1,2}（Java: removeAll）


## 4. tuple（不可变序列，Java 里没有等价核心类型）


**tuple 是不可变的有序序列**，Java 没有直接对应的类型。

核心特点：
- 用 `()` 创建，创建后不能修改（不能 append、remove）
- 适合表达**固定结构**：坐标 `(x, y)`、函数返回多个值、配置常量
- 比 list 性能略好，因为不可变
- 最重要的用途：配合**解包**语法，让函数优雅地返回多个值

In [ ]:
# 说明：本段示例对应「4. tuple（不可变序列，Java 里没有等价核心类型）」，建议先看注释再运行，重点观察输出与预期是否一致。

point = (10, 20)
print(point[0])

# tuple 不可变，适合表达固定结构的返回值
user = ("kai", 18, True)
name, age, is_dev = user
print(name, age, is_dev)


In [ ]:
# 说明：本段示例对应「4. tuple（不可变序列，Java 里没有等价核心类型）」，建议先看注释再运行，重点观察输出与预期是否一致。

def parse_user(raw: str):
    name, age = raw.split(",")
    return name, int(age)

name, age = parse_user("kai,18")
print(name, age)


## 5. 解包（Unpacking）


**解包（Unpacking）是 Python 独有的语法糖**，Java 完全没有等价写法。

核心特点：
- 把序列中的值**一次性赋给多个变量**，代码更简洁
- `*` 可以收集"剩余元素"（类比 JS 的 `...rest`）
- `**` 可以展开 dict（类比 JS 的 `{...obj}`）
- 函数返回 tuple 时，调用方用解包接收，让多返回值变得自然

In [ ]:
# 说明：演示解包与交换赋值，减少临时变量，让代码更简洁。

# 基本解包
a, b = 1, 2
a, b = b, a   # 交换变量，Java 需要临时变量
print(a, b)

# [Review] * 收集剩余元素是 Java 完全没有的语法，需要解释
# *middle 会把"剩余的元素"收集成一个 list（类比 JS 的 ...rest）
first, *middle, last = [10, 20, 30, 40, 50]
print(first, middle, last)  # 10 [20, 30, 40] 50


In [ ]:
# 说明：演示用 ** 展开并合并 dict，常用于拼装配置或请求参数。

# [Review] ** 展开 dict 是 Java 没有的语法，类比 JS 的 {...obj1, ...obj2}
profile = {"name": "kai", "age": 18}
extra = {"city": "shanghai"}
merged = {**profile, **extra}  # 把两个 dict 展开后合并成一个新 dict
print(merged)

# ⚠️ 坑：两个 dict 有相同 key 时，后者静默覆盖前者，不会报错也不会提示
profile2 = {"name": "kai", "age": 18}
override = {"age": 99}               # age 和 profile2 冲突
merged2 = {**profile2, **override}
print(merged2)  # {'name': 'kai', 'age': 99}  age 被覆盖了


## 6. 常见类型转换


**四种类型之间可以互相转换**，用各自的构造函数即可。

常用场景：
- `list → set`：去重
- `set → list`：去重后需要排序或索引访问
- `list of tuples → dict`：键值对列表转字典
- `dict.keys() / .values()`：返回的是视图对象，需要用 `list()` 转换才能索引

In [ ]:
# 说明：本段示例对应「6. 常见类型转换」，建议先看注释再运行，重点观察输出与预期是否一致。

nums = [1, 2, 2, 3]

as_set = set(nums)
as_tuple = tuple(nums)
back_to_list = list(as_set)

print(as_set)
print(as_tuple)
print(back_to_list)


In [ ]:
# 说明：本段示例对应「6. 常见类型转换」，建议先看注释再运行，重点观察输出与预期是否一致。

pairs = [("name", "kai"), ("age", 18)]
info = dict(pairs)
print(info)

keys = list(info.keys())
values = list(info.values())
print(keys)
print(values)


## 7. Java 对照速记

| Java | Python |
|---|---|
| `List<T>` | `list` |
| `Map<K, V>` | `dict` |
| `Set<T>` | `set` |
| 需手写循环拷贝子数组 | 切片 `arr[a:b:c]` |
| `map.getOrDefault(k, v)` | `dict.get(k, v)` |
| `Collections.unmodifiableList(...)` | `tuple`（不可变序列） |


## 今日打卡题

请独立完成后再看答案：

给定：

`words = ["ai", "python", "ai", "java", "python", "ai"]`

要求：
1. 统计每个单词出现次数（dict）
2. 输出去重后的单词集合（set）
3. 把统计结果按出现次数倒序输出（list）


In [ ]:
# 说明：这是练习留空区域，建议先独立完成，再运行后续参考答案进行对照。

words = ["ai", "python", "ai", "java", "python", "ai"]

# TODO: 先自己写


In [ ]:
# 说明：演示用 dict.get(key, default) 做计数聚合，避免首次出现 key 时抛异常。

words = ["ai", "python", "ai", "java", "python", "ai"]

counter = {}
for w in words:
    counter[w] = counter.get(w, 0) + 1

unique_words = set(words)

# [Review] lambda 会在 03-functions 详细讲，这里先知道它是一个匿名小函数
# key=lambda x: x[1] 意思是：按每个元素的第二个值（出现次数）排序
sorted_items = sorted(counter.items(), key=lambda x: x[1], reverse=True)

print("counter =", counter)
print("unique_words =", unique_words)
print("sorted_items =", sorted_items)


下一步建议：完成 `03-functions.ipynb`，把 `*args / **kwargs / decorator` 跑通。
